# Construction Project Delay Analysis
**Industry:** Construction and Infrastructure  
**Question:** Why are construction projects getting delayed?

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Paths
DATA_PATH = '../data/project_data.csv'
VIZ_DIR   = '../outputs/visualizations'
MDL_DIR   = '../outputs/models'
RPT_DIR   = '../outputs/reports'
os.makedirs(VIZ_DIR, exist_ok=True)
os.makedirs(MDL_DIR, exist_ok=True)
os.makedirs(RPT_DIR, exist_ok=True)
print('Setup complete.')

## 2. Load & Clean Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print('\nData types:')
print(df.dtypes)

In [ ]:
# Strip whitespace
for col in df.select_dtypes('object').columns:
    df[col] = df[col].str.strip()

# TARGET ENGINEERING RULE:
# Progress_Gap = Planned_Progress - Actual_Progress
# Delayed = 1 if Progress_Gap > 10 pp, else 0
df['Progress_Gap'] = df['Planned_Progress'] - df['Actual_Progress']
df['Delayed'] = (df['Progress_Gap'] > 10).astype(int)

print('Delay target distribution:')
print(df['Delayed'].value_counts())
print(f'\nDelay Rate: {df["Delayed"].mean()*100:.1f}%')
df[['Progress_Gap','Delayed']].describe()

## 3. Descriptive Statistics

In [ ]:
df.describe().round(2)

In [ ]:
for col in ['Project_Category','Project_Size','Material_Delivery','Weather','Contractor','Milestone']:
    print(f'{col}:\n{df[col].value_counts()}\n')

## 4. Visualization 1 — Planned vs Actual Progress

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df))
ax.plot(x, df['Planned_Progress'], label='Planned Progress', color='steelblue', lw=1.5)
ax.plot(x, df['Actual_Progress'],  label='Actual Progress',  color='tomato',    lw=1.5)
ax.fill_between(x, df['Planned_Progress'], df['Actual_Progress'],
                alpha=0.2, color='orange', label='Gap')
ax.set_xlabel('Project Index')
ax.set_ylabel('Progress (%)')
ax.set_title('Planned vs Actual Progress Across All Projects')
ax.legend()
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/planned_vs_actual_progress.png', dpi=150)
plt.show()

## 5. Visualization 2 — Delay by Project Category

In [ ]:
cat_delay = df.groupby('Project_Category')['Progress_Gap'].mean().sort_values(ascending=False)
print(cat_delay)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cat_delay.index, cat_delay.values,
              color=['#e74c3c','#e67e22','#3498db','#2ecc71'])
ax.set_xlabel('Project Category')
ax.set_ylabel('Average Progress Gap (%)')
ax.set_title('Average Delay (Progress Gap) by Project Category')
for bar, val in zip(bars, cat_delay.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{val:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/delay_by_project_category.png', dpi=150)
plt.show()

## 6. Visualization 3 — Material Delivery vs Project Delay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column='Progress_Gap', by='Material_Delivery', ax=axes[0])
axes[0].set_title('Progress Gap by Material Delivery')
axes[0].set_xlabel('Material Delivery')
axes[0].set_ylabel('Progress Gap (%)')
plt.sca(axes[0])
plt.title('Progress Gap by Material Delivery')

delay_rate = df.groupby('Material_Delivery')['Delayed'].mean() * 100
axes[1].bar(delay_rate.index, delay_rate.values, color=['#27ae60','#e74c3c'])
axes[1].set_title('% Delayed Projects by Material Delivery')
axes[1].set_xlabel('Material Delivery')
axes[1].set_ylabel('% Projects Delayed')
for i,(idx,val) in enumerate(delay_rate.items()):
    axes[1].text(i, val+0.5, f'{val:.1f}%', ha='center', fontsize=11)

plt.suptitle('')
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/material_delivery_vs_delay.png', dpi=150)
plt.show()

## 7. Delay Analysis — All Dimensions

In [ ]:
print('=== By Milestone ===')
print(df.groupby('Milestone').agg(
    Avg_Gap=('Progress_Gap','mean'), Delay_Rate=('Delayed','mean'), Count=('Project_ID','count')
).round(2).sort_values('Avg_Gap', ascending=False))

print('\n=== By Contractor ===')
print(df.groupby('Contractor').agg(
    Avg_Gap=('Progress_Gap','mean'), Delay_Rate=('Delayed','mean'), Count=('Project_ID','count')
).round(2).sort_values('Avg_Gap', ascending=False))

print('\n=== By Weather ===')
print(df.groupby('Weather').agg(
    Avg_Gap=('Progress_Gap','mean'), Delay_Rate=('Delayed','mean'), Count=('Project_ID','count')
).round(2).sort_values('Avg_Gap', ascending=False))

print('\n=== By Project Size ===')
print(df.groupby('Project_Size').agg(
    Avg_Gap=('Progress_Gap','mean'), Delay_Rate=('Delayed','mean'), Count=('Project_ID','count')
).round(2).sort_values('Avg_Gap', ascending=False))

## 8. High-Risk Projects

In [ ]:
high_risk = df[df['Progress_Gap'] >= 15].sort_values('Progress_Gap', ascending=False)
print(f'High-risk projects (Gap >= 15%): {len(high_risk)}')
high_risk[['Project_ID','Project_Category','Contractor','Progress_Gap','Material_Delivery','Weather']].head(15)

## 9. Random Forest Model

In [ ]:
CATEGORICAL_COLS = ['Project_Category','Project_Size','Material_Delivery','Weather','Contractor','Milestone']
NUMERICAL_COLS   = ['Budget','Workforce','Planned_Progress']

df_enc = df.copy()
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_enc[col+'_enc'] = le.fit_transform(df_enc[col])
    encoders[col] = le

feature_cols = NUMERICAL_COLS + [c+'_enc' for c in CATEGORICAL_COLS]
X = df_enc[feature_cols]
y = df_enc['Delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
print('Model trained.')

In [ ]:
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['On Track','Delayed']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm, display_labels=['On Track','Delayed']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/rf_confusion_matrix.png', dpi=150)
plt.show()

## 10. Visualization 4 — Feature Importance (Early Warning Indicators)

In [ ]:
rename_map = {
    'Budget':'Budget','Workforce':'Workforce','Planned_Progress':'Planned Progress',
    'Project_Category_enc':'Project Category','Project_Size_enc':'Project Size',
    'Material_Delivery_enc':'Material Delivery','Weather_enc':'Weather',
    'Contractor_enc':'Contractor','Milestone_enc':'Milestone'
}
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.index = [rename_map.get(i,i) for i in importances.index]

fig, ax = plt.subplots(figsize=(9,6))
colors = ['#e74c3c' if v >= importances.quantile(0.75) else '#3498db' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Random Forest — Feature Importance\n(Early Warning Indicators)', fontsize=13)
ax.set_xlabel('Importance Score')
ax.axvline(importances.mean(), color='gray', linestyle='--', label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/feature_importance.png', dpi=150)
plt.show()

print('\nFeature Importance (sorted):')
print(importances.sort_values(ascending=False))

In [ ]:
# Save model
model_path = f'{MDL_DIR}/random_forest_model.pkl'
joblib.dump({'model': rf, 'encoders': encoders, 'features': feature_cols}, model_path)
print(f'Model saved: {model_path}')

## 11. Key Insights

1. **Material delivery is the #1 delay driver** — 100% delay rate when delayed, 0% when on time.
2. **Weather is decisive** — Stormy/Rainy projects have 100% delay rates; Clear weather = 0%.
3. **Industrial projects are highest risk** — 100% delay rate, avg gap 16.64%, highest budgets.
4. **Contractor performance is polarised** — ContractorA/C: 0% delay; ContractorB/D: 100% delay.
5. **Large projects are 1.7× more likely to be delayed** than small ones (59% vs 34%).
6. **Structural milestones are the most vulnerable phase** — 53% delay rate.
7. **Random Forest achieves 100% accuracy** — delay is fully predictable from observable project attributes.

## 12. Practical Action Plan

- **Audit material supply chains** — establish backup suppliers for all critical materials.
- **Review ContractorB and ContractorD** — implement performance improvement plans.
- **Weather-contingency scheduling** — build buffer time into Structural milestones.
- **Deploy early warning dashboards** — flag projects the moment Material_Delivery = Delayed.
- **Use the RF model at project initiation** to score delay risk before construction begins.
- **Standardise contractor performance metrics** across all projects.
- **Milestone-based progress reviews** with mandatory corrective action if gap > 5 pp.